# Notebook 42 — WorkingMemory ↔ LLMAgent Wiring

This notebook demonstrates how `WorkingMemory` integrates with `LLMAgent` to
automatically inject a rolling conversation context into prompts.

**Core mechanism:** `LLMAgent._render()` calls `_inject_working_memory()` before
formatting the prompt template. The memory window is placed under the key
`_memory_context` (configurable) in the context dict, making it available as
`{_memory_context}` inside any prompt template.

| Component | Role |
|---|---|
| `WorkingMemory` | Bounded sliding window of recent context items |
| `LLMAgent` | Accepts `working_memory=` param; injects memory before rendering |
| `MemoryAgent` | Stateful wrapper that accumulates call history |
| `Chain` | Sequential pipeline connecting MemoryAgent + LLMAgent |

> All examples use mock functions — no real LLM calls are made.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../sdk')

from multigen.memory import WorkingMemory
from multigen.agent import LLMAgent, MemoryAgent, FunctionAgent
from multigen.chain import Chain

print('Modules loaded successfully')

## Section 1: WorkingMemory Recap

`WorkingMemory` is a bounded deque that holds the most recent N context items.
Items can be anything — dicts, strings, dataclass instances.
When it reaches `maxsize`, the oldest item is automatically evicted.

In [ ]:
# Create a WorkingMemory with a window of 5 items
wm = WorkingMemory(maxsize=5)

print(f'Created WorkingMemory(maxsize=5)')
print(f'Initial size: {len(wm)}')

In [ ]:
# Push conversation turns as dicts
turn_history = [
    {'role': 'user',      'content': 'What is a neural network?'},
    {'role': 'assistant', 'content': 'A neural network is a system of interconnected nodes.'},
    {'role': 'user',      'content': 'How does backpropagation work?'},
    {'role': 'assistant', 'content': 'Backpropagation computes gradients via the chain rule.'},
    {'role': 'user',      'content': 'What is gradient descent?'},
    {'role': 'assistant', 'content': 'Gradient descent minimises the loss by iterative updates.'},
    # This 7th push will evict the first two items (maxsize=5)
    {'role': 'user',      'content': 'What is overfitting?'},
]

for turn in turn_history:
    wm.push(turn)

print(f'Pushed {len(turn_history)} items into maxsize=5 window')
print(f'Current window size: {len(wm)}  (oldest items were evicted)')

In [ ]:
# .all() returns the entire current window
print('Current window contents:')
for i, item in enumerate(wm.all()):
    print(f'  [{i}] {item["role"]:10s}: {item["content"]}')

print()
# .peek(n) returns the last N items
print('Last 2 items (peek):')
for item in wm.peek(2):
    print(f'  {item["role"]:10s}: {item["content"]}')

In [ ]:
# .summarize() produces a newline-joined string representation
# useful as a raw text block for prompt injection
print('wm.summarize() output:')
print('=' * 60)
print(wm.summarize())
print('=' * 60)
print(f'\n(This string gets injected as {{_memory_context}} in LLMAgent prompts)')

## Section 2: LLMAgent with working_memory Parameter

`LLMAgent` accepts a `working_memory` parameter at construction time.
When set, `_render()` calls `_inject_working_memory()` which:
1. Calls `wm.window()` (or `wm.all()` as fallback) to get recent items
2. Formats each item as a `- content` bullet
3. Stores the result under `ctx[working_memory_key]` (default: `_memory_context`)
4. Then formats the prompt template with the enriched context

This means your template just needs `{_memory_context}` to receive memory.

In [ ]:
# Create a WorkingMemory with some conversation history
conversation_wm = WorkingMemory(maxsize=6)

conversation_wm.push({'role': 'user',      'content': 'I am working on a Python web scraper'})
conversation_wm.push({'role': 'assistant', 'content': 'I can help with that. What library are you using?'})
conversation_wm.push({'role': 'user',      'content': 'I am using BeautifulSoup and requests'})
conversation_wm.push({'role': 'assistant', 'content': 'Good choice. What data are you trying to extract?'})

print(f'Conversation memory has {len(conversation_wm)} turns')

In [ ]:
# LLMAgent with working_memory wired in at construction time
# The prompt template includes {_memory_context} to receive the injected memory
agent = LLMAgent(
    'assistant',
    prompt=(
        'You are a helpful assistant.\n'
        '\nConversation so far:\n{_memory_context}\n'
        '\nUser: {user_input}\n'
        'Assistant:'
    ),
    working_memory=conversation_wm,
    # working_memory_key defaults to '_memory_context'
)

print(f'LLMAgent created: {agent.name}')
print(f'working_memory   : {type(agent._working_memory).__name__}')
print(f'memory_key       : {agent._working_memory_key!r}')

In [ ]:
# Call _render() directly to see how memory is injected into the prompt
# (without making a real LLM API call)
ctx = {'user_input': 'How do I handle pagination?'}

rendered_prompt = agent._render(ctx)

print('Rendered prompt with injected working memory:')
print('=' * 65)
print(rendered_prompt)
print('=' * 65)

In [ ]:
# As memory grows, older items roll off the window
conversation_wm.push({'role': 'user',      'content': 'How do I handle pagination?'})
conversation_wm.push({'role': 'assistant', 'content': 'Use the ?page=N query parameter and loop until empty.'})
conversation_wm.push({'role': 'user',      'content': 'What about rate limiting?'})

# Window is now full (maxsize=6) — oldest entries evicted
print(f'Memory window after more turns: {len(conversation_wm)} items')

next_ctx = {'user_input': 'Should I use asyncio for speed?'}
next_rendered = agent._render(next_ctx)

print('\nUpdated rendered prompt (older turns evicted):')
print('=' * 65)
print(next_rendered)
print('=' * 65)

## Section 3: Context-Key Wiring

Alternatively, instead of passing `working_memory=` at construction time, you
can inject memory *dynamically* by placing a `WorkingMemory` instance under
`ctx['_working_memory']`. `LLMAgent._inject_working_memory()` checks this key
first and falls back to the constructor-bound instance.

This pattern is useful when:
- Multiple agents share a single memory object passed through a pipeline
- You want to swap memory per-call without recreating the agent

In [ ]:
# Agent with NO constructor-bound working memory
agent_no_wm = LLMAgent(
    'context_agent',
    prompt=(
        'Context from memory:\n{_memory_context}\n\n'
        'Task: {task}'
    ),
    # working_memory not set — will use ctx dict injection
)

print(f'Agent created without constructor-bound memory: {agent_no_wm.name}')
print(f'_working_memory attr: {agent_no_wm._working_memory}')

In [ ]:
# Build a session memory
session_wm = WorkingMemory(maxsize=4)
session_wm.push('User is a data scientist')
session_wm.push('Working on tabular data analysis')
session_wm.push('Prefers Python with pandas')

# Pass memory via the ctx dict using the '_working_memory' key
ctx_with_memory = {
    'task': 'Suggest a visualisation approach for outlier detection',
    '_working_memory': session_wm,   # <-- dynamic injection
}

rendered = agent_no_wm._render(ctx_with_memory)

print('Rendered prompt via ctx-key injection:')
print('=' * 65)
print(rendered)
print('=' * 65)

In [ ]:
# Custom working_memory_key — use any template placeholder name
agent_custom_key = LLMAgent(
    'custom_key_agent',
    prompt='Background:\n{session_context}\n\nQuestion: {question}',
    working_memory=session_wm,
    working_memory_key='session_context',   # custom placeholder name
)

rendered_custom = agent_custom_key._render({'question': 'What visualisation library should I use?'})

print('Rendered with custom memory key {session_context}:')
print('=' * 65)
print(rendered_custom)
print('=' * 65)

## Section 4: Full Pipeline — Chain with MemoryAgent + LLMAgent

A typical production pattern chains a `MemoryAgent` (which accumulates history)
before an `LLMAgent` that reads from `WorkingMemory`. The memory propagates
through the pipeline ctx dict automatically.

In [ ]:
# Shared WorkingMemory instance — both agents see the same window
pipeline_wm = WorkingMemory(maxsize=8)

# A mock LLM backend that echoes the injected memory context
async def mock_llm_backend(ctx):
    """Returns a mock response without calling any external API."""
    prompt = ctx.get('_rendered_prompt', '')
    user_input = ctx.get('user_input', '')
    return {**ctx, 'response': f'[MOCK RESPONSE to: "{user_input[:40]}"]'}

print('Pipeline setup ready')

In [ ]:
# MemoryAgent accumulates turn history in its own internal list
# We also manually push to pipeline_wm so LLMAgent can read it

async def memory_push_agent(ctx):
    """Pushes current turn into WorkingMemory before passing context downstream."""
    user_input = ctx.get('user_input', '')
    if user_input:
        pipeline_wm.push({'role': 'user', 'content': user_input})
    return ctx

memory_pusher = FunctionAgent('memory_pusher', fn=memory_push_agent)

print(f'Memory pusher agent created: {memory_pusher.name}')

In [ ]:
# LLMAgent that reads from pipeline_wm
# We mock the actual LLM call by overriding run() via FunctionAgent

memory_aware_template = (
    'Conversation history:\n{_memory_context}\n\n'
    'Current question: {user_input}\n\n'
    'Answer:'
)

# We'll simulate _render manually to demonstrate injection without a real API key
response_agent_llm = LLMAgent(
    'memory_aware_llm',
    prompt=memory_aware_template,
    working_memory=pipeline_wm,
)

print(f'Memory-aware LLMAgent created: {response_agent_llm.name}')

In [ ]:
# Simulate 3 turns of the pipeline
turn_inputs = [
    'What is a transformer architecture?',
    'How does multi-head attention work?',
    'Can you compare it to RNNs?',
]

for turn_idx, user_input in enumerate(turn_inputs):
    print(f'\n--- Turn {turn_idx + 1}: "{user_input}" ---')

    # Step 1: push to memory
    pipeline_wm.push({'role': 'user', 'content': user_input})

    ctx = {'user_input': user_input, '_working_memory': pipeline_wm}

    # Step 2: render prompt with injected memory (no real LLM call)
    rendered = response_agent_llm._render(ctx)

    print('Rendered prompt:')
    # Show just the memory context section
    lines = rendered.split('\n')
    for line in lines:
        print(f'  {line}')

    # Step 3: simulate assistant response and push to memory
    mock_reply = f'[MOCK] Answer about: {user_input[:35]}...'
    pipeline_wm.push({'role': 'assistant', 'content': mock_reply})

print(f'\nFinal memory window size: {len(pipeline_wm)}')

In [ ]:
# Demonstrate Chain with memory propagation
# Chain passes ctx through agents sequentially, merging outputs

fresh_wm = WorkingMemory(maxsize=4)

async def inject_memory_step(ctx):
    """Agent step that injects working memory into ctx for downstream agents."""
    fresh_wm.push({'turn': ctx.get('user_input', '')})
    return {**ctx, '_working_memory': fresh_wm}

async def render_step(ctx):
    """Agent step that renders the memory-enriched prompt."""
    # Simulate what LLMAgent._render() does
    wm = ctx.get('_working_memory')
    if wm:
        window = wm.all()
        memory_text = '\n'.join(
            f"- {e}" if isinstance(e, str) else f"- {e.get('content', str(e))}"
            for e in window
        )
    else:
        memory_text = '(empty)'
    template = 'Memory:\n{_memory_context}\n\nQ: {user_input}'
    rendered = template.format(_memory_context=memory_text, **ctx)
    return {**ctx, 'rendered_prompt': rendered}

injector = FunctionAgent('memory_injector', fn=inject_memory_step)
renderer = FunctionAgent('prompt_renderer',  fn=render_step)

chain = Chain([injector, renderer])

# Run one turn
result = await chain({'user_input': 'Explain self-attention'})

print('Chain output - rendered_prompt field:')
print('=' * 65)
print(result.get('rendered_prompt', ''))
print('=' * 65)

## Section 5: Best Practices — Template Patterns for Memory Context

In [ ]:
# Pattern A: Simple append — memory at the bottom just before the question
pattern_a = LLMAgent(
    'pattern_a',
    prompt='Answer the user question.\n\n{_memory_context}\n\nQuestion: {question}',
    working_memory=WorkingMemory(maxsize=5),
)

# Pattern B: System-style header — memory framed as "prior context"
pattern_b = LLMAgent(
    'pattern_b',
    prompt=(
        'You are a knowledgeable assistant.\n\n'
        'Prior conversation context:\n'
        '---\n'
        '{_memory_context}\n'
        '---\n\n'
        'New question: {question}\n\n'
        'Answer:'
    ),
    working_memory=WorkingMemory(maxsize=10),
)

# Pattern C: Minimal — memory key renamed to {ctx} for shorter templates
pattern_c = LLMAgent(
    'pattern_c',
    prompt='Context: {ctx}\n\nTask: {task}',
    working_memory=WorkingMemory(maxsize=3),
    working_memory_key='ctx',
)

print('Three template patterns defined')

In [ ]:
# Seed all three with the same working memory content
for agent_obj in [pattern_a, pattern_b, pattern_c]:
    agent_obj._working_memory.push('User is a senior ML engineer')
    agent_obj._working_memory.push('Focused on production LLM deployment')
    agent_obj._working_memory.push('Prefers concise technical answers')

q_ctx = {'question': 'What is KV-cache and why does it matter?', 'task': 'Explain KV-cache benefits'}

print('Pattern A — simple append:')
print(pattern_a._render(q_ctx))
print()
print('Pattern B — system-style header:')
print(pattern_b._render(q_ctx))
print()
print('Pattern C — minimal, custom key:')
print(pattern_c._render({'task': 'Explain KV-cache benefits'}))

In [ ]:
# Graceful handling: if memory is empty, {_memory_context} renders as empty string
empty_wm_agent = LLMAgent(
    'fresh_agent',
    prompt='Context:\n{_memory_context}\n\nQuestion: {question}',
    working_memory=WorkingMemory(maxsize=5),  # empty
)

rendered_empty = empty_wm_agent._render({'question': 'What is a large language model?'})
print('Rendered with empty WorkingMemory:')
print(repr(rendered_empty))
print()
print('Full output:')
print(rendered_empty)

In [ ]:
# Safety: if template does NOT include {_memory_context}, memory is injected into
# ctx but silently ignored during format() — no KeyError
agent_no_placeholder = LLMAgent(
    'safe_agent',
    prompt='Just answer: {question}',  # no {_memory_context} placeholder
    working_memory=WorkingMemory(maxsize=5),
)
agent_no_placeholder._working_memory.push('Some prior context')

# This renders without error — memory is injected into ctx but not used in template
result = agent_no_placeholder._render({'question': 'What is PyTorch?'})
print(f'Safe render (no placeholder): {result!r}')
print('No KeyError — memory injection is non-destructive')

## Summary

```python
from multigen.memory import WorkingMemory
from multigen.agent import LLMAgent

# 1. Constructor wiring (memory fixed to this agent)
wm = WorkingMemory(maxsize=10)
agent = LLMAgent(
    'assistant',
    prompt='Context:\n{_memory_context}\n\nQ: {question}',
    working_memory=wm,
)
wm.push({'role': 'user', 'content': 'Hello'})
prompt = agent._render({'question': 'How are you?'})
# --> {_memory_context} is filled with the window contents

# 2. Context-key wiring (memory passed per-call)
agent2 = LLMAgent('assistant2', prompt='Context:\n{_memory_context}\n\nQ: {question}')
result = await agent2({'question': '...', '_working_memory': wm})

# 3. Custom memory key
agent3 = LLMAgent(
    'assistant3',
    prompt='Background: {session_ctx}\n\nQ: {question}',
    working_memory=wm,
    working_memory_key='session_ctx',
)
```

**Best practices:**
- Use `maxsize=6–10` for conversational context; larger values dilute relevance
- Always include `{_memory_context}` (or your custom key) in the template
- Push both user turns and assistant replies to maintain symmetry
- For pipeline memory, pass `_working_memory` via ctx dict to decouple agents
- Missing placeholder is safe — memory enriches ctx but does not cause errors